# 🤖 AutoGen AI Workflow: Code Generation, Review & Save

This notebook implements a **multi-agent AutoGen workflow** that:
1. 📥 Accepts a user query or dataset
2. 🧠 **Code Generator Agent** writes analysis/processing code
3. 🔍 **Code Reviewer Agent** checks and improves the code
4. 💾 **Saves output** as CSV or JSON

---
### Architecture
```
User Query / Dataset
        │
        ▼
┌──────────────────┐      review & fix      ┌──────────────────┐
│  CodeGenerator   │ ◄────────────────────► │  CodeReviewer    │
│     Agent        │                        │     Agent        │
└──────────────────┘                        └──────────────────┘
        │
        ▼
┌──────────────────┐
│   UserProxy /    │
│  Executor Agent  │
└──────────────────┘
        │
        ▼
  Save as CSV / JSON
```

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install pyautogen openai pandas

## 2. Imports & Configuration

In [ ]:
import autogen
import pandas as pd
import json
import os
import re
from pathlib import Path

print(f"AutoGen version: {autogen.__version__}")

## 3. LLM Configuration

> **Set your API key below.** Supports OpenAI, Azure OpenAI, or any OpenAI-compatible endpoint (e.g. Ollama, LiteLLM).

In [ ]:
import os

# ── Configure your LLM here ──────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "sk-..."   # Replace with your actual key

LLM_CONFIG = {
    "config_list": [
        {
            "model": "gpt-4o",             # Change model if needed
            "api_key": os.environ["OPENAI_API_KEY"],
        }
    ],
    "temperature": 0.2,
    "cache_seed": 42,                      # Reproducible runs; set None to disable
}
# ─────────────────────────────────────────────────────────────────────────────

print("✅ LLM config ready.")

## 4. Define the Agents

In [ ]:
# ── Agent 1: Code Generator ───────────────────────────────────────────────────
code_generator = autogen.AssistantAgent(
    name="CodeGenerator",
    llm_config=LLM_CONFIG,
    system_message="""
You are an expert Python data scientist.
When given a user query or dataset description, you write clean, well-commented
Python code that:
  1. Loads or generates the data (using pandas where appropriate).
  2. Performs the requested analysis or transformation.
  3. Saves the final result BOTH as a CSV file ('output.csv') AND as a JSON
     file ('output.json') using the helper functions below.

Always wrap ALL executable code in a single ```python ... ``` block.
After writing the code, say TERMINATE.
""",
)

# ── Agent 2: Code Reviewer ────────────────────────────────────────────────────
code_reviewer = autogen.AssistantAgent(
    name="CodeReviewer",
    llm_config=LLM_CONFIG,
    system_message="""
You are a senior Python code reviewer.
When CodeGenerator provides code, you must:
  1. Check for correctness, edge cases, missing imports, and potential errors.
  2. Verify that the code saves output BOTH as 'output.csv' AND 'output.json'.
  3. If the code is correct, reply with: LGTM. TERMINATE
  4. If issues are found, provide the fully corrected code in a single
     ```python ... ``` block and then say TERMINATE.
""",
)

# ── Agent 3: User Proxy / Executor ────────────────────────────────────────────
user_proxy = autogen.UserProxyAgent(
    name="UserProxy",
    human_input_mode="NEVER",          # Fully automated – change to ALWAYS for interactive
    max_consecutive_auto_reply=10,
    code_execution_config={
        "work_dir": "autogen_workspace",  # Code runs here
        "use_docker": False,
    },
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),
)

print("✅ Agents created: CodeGenerator, CodeReviewer, UserProxy")

## 5. Helper: Save Results to CSV & JSON

In [ ]:
import os, json, pandas as pd
from pathlib import Path

WORKSPACE = Path("autogen_workspace")
WORKSPACE.mkdir(exist_ok=True)

def save_results(data, filename_stem="output"):
    """
    Save data (dict, list, or DataFrame) as both CSV and JSON.
    Returns paths to the saved files.
    """
    csv_path  = WORKSPACE / f"{filename_stem}.csv"
    json_path = WORKSPACE / f"{filename_stem}.json"

    # ── Normalise to DataFrame ────────────────────────────────────
    if isinstance(data, pd.DataFrame):
        df = data
    elif isinstance(data, (list, dict)):
        df = pd.DataFrame(data if isinstance(data, list) else [data])
    else:
        raise TypeError(f"Unsupported data type: {type(data)}")

    # ── Save ──────────────────────────────────────────────────────
    df.to_csv(csv_path,  index=False)
    df.to_json(json_path, orient="records", indent=2)

    print(f"✅ Saved CSV  → {csv_path}")
    print(f"✅ Saved JSON → {json_path}")
    return str(csv_path), str(json_path)


def load_and_preview(csv_path):
    """Quick preview of the generated CSV."""
    df = pd.read_csv(csv_path)
    print(f"\n📊 Shape: {df.shape}")
    return df.head(10)


print("✅ Helper functions ready.")

## 6. Build the Group Chat Workflow

In [ ]:
def run_autogen_workflow(user_query: str):
    """
    Run the full AutoGen workflow:
      UserProxy → CodeGenerator → CodeReviewer → Execute → Save CSV + JSON

    Parameters
    ----------
    user_query : str
        A natural-language description of the task or a path to a dataset.
    """
    print("=" * 65)
    print("🚀  AutoGen Workflow Starting")
    print("=" * 65)
    print(f"📝 Query: {user_query}\n")

    # ── GroupChat: Generator ↔ Reviewer, managed by UserProxy ─────
    groupchat = autogen.GroupChat(
        agents=[user_proxy, code_generator, code_reviewer],
        messages=[],
        max_round=12,
        speaker_selection_method="round_robin",  # Generator → Reviewer → Executor
    )

    manager = autogen.GroupChatManager(
        groupchat=groupchat,
        llm_config=LLM_CONFIG,
    )

    # ── Kick off the conversation ─────────────────────────────────
    user_proxy.initiate_chat(
        manager,
        message=(
            f"{user_query}\n\n"
            "Make sure the final code saves the result as both "
            "'output.csv' and 'output.json' in the current directory."
        ),
    )

    # ── Post-run: check for generated files ───────────────────────
    print("\n" + "=" * 65)
    print("📂  Checking generated output files...")
    csv_path  = WORKSPACE / "output.csv"
    json_path = WORKSPACE / "output.json"

    results = {}
    if csv_path.exists():
        results["csv"] = str(csv_path)
        print(f"✅ CSV  found → {csv_path}")
    else:
        print("⚠️  output.csv not found.")

    if json_path.exists():
        results["json"] = str(json_path)
        print(f"✅ JSON found → {json_path}")
    else:
        print("⚠️  output.json not found.")

    return results


print("✅ Workflow function ready.")

## 7. Run the Workflow

Edit the **`USER_QUERY`** below — either a natural-language task or a path to your dataset.

In [ ]:
# ── Example 1: Natural-language query ────────────────────────────────────────
USER_QUERY = """
Generate a synthetic sales dataset with 100 rows containing columns:
product_name, category, units_sold, unit_price, revenue, sale_date.
Calculate total revenue per category and save the summary.
"""

# ── Example 2: Analyse an existing file (uncomment to use) ───────────────────
# USER_QUERY = """
# Load the CSV at 'my_data.csv', compute descriptive statistics for all
# numeric columns, identify the top 5 rows by the first numeric column,
# and save the results.
# """

# ── Run ──────────────────────────────────────────────────────────────────────
output_files = run_autogen_workflow(USER_QUERY)

## 8. Preview Generated Outputs

In [ ]:
# ── Preview CSV ───────────────────────────────────────────────────────────────
if "csv" in output_files:
    print("📄 CSV Preview:")
    df_result = pd.read_csv(output_files["csv"])
    print(f"  Shape: {df_result.shape}")
    display(df_result.head(10))
else:
    print("No CSV output to preview.")

In [ ]:
# ── Preview JSON ──────────────────────────────────────────────────────────────
if "json" in output_files:
    print("📄 JSON Preview (first 5 records):")
    with open(output_files["json"]) as f:
        records = json.load(f)
    print(json.dumps(records[:5], indent=2))
else:
    print("No JSON output to preview.")

## 9. (Optional) Custom Save — Use Your Own Data

In [ ]:
# ── Bring your own DataFrame and save it with one call ────────────────────────
# Uncomment and adapt as needed:

# my_df = pd.DataFrame({
#     "name":  ["Alice", "Bob", "Carol"],
#     "score": [92, 85, 78],
# })
# csv_p, json_p = save_results(my_df, filename_stem="my_custom_output")
# display(load_and_preview(csv_p))

print("Uncomment the block above to save your own data.")

## 10. Extract & Inspect the Generated Code

In [ ]:
def extract_last_code_block(groupchat_messages):
    """
    Walk backwards through the GroupChat messages and return
    the last Python code block found.
    """
    pattern = re.compile(r"```python\n(.*?)```", re.DOTALL)
    for msg in reversed(groupchat_messages):
        content = msg.get("content", "") or ""
        matches = pattern.findall(content)
        if matches:
            return matches[-1].strip()
    return "No code block found in chat history."

# Retrieve and print the final reviewed code
try:
    final_code = extract_last_code_block(groupchat.messages)
    print("📝 Final reviewed code:\n")
    print(final_code)
except NameError:
    print("Run Section 7 first to populate the chat history.")

---

## Workflow Summary

| Step | Agent | Action |
|------|-------|--------|
| 1 | **UserProxy** | Sends query / dataset to the group chat |
| 2 | **CodeGenerator** | Writes Python analysis code + save logic |
| 3 | **CodeReviewer** | Reviews, fixes bugs, confirms CSV & JSON saving |
| 4 | **UserProxy** | Executes the approved code in `autogen_workspace/` |
| 5 | — | `output.csv` and `output.json` are written to disk |

### Customisation Tips
- **Model**: change `"model"` in `LLM_CONFIG` to `gpt-4o-mini`, `claude-*`, or any local model via LiteLLM.
- **Rounds**: increase `max_round` for complex multi-step tasks.
- **Human input**: set `human_input_mode="ALWAYS"` in `UserProxy` to approve each step interactively.
- **Docker sandbox**: set `"use_docker": True` for isolated code execution.